<a href="https://colab.research.google.com/github/jonaire-tate/dx702-experimental-design/blob/main/DX702_CodingQuiz_Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX702 Coding Quiz Week 1
## Regression and Matching
Jonaire Tate | Boston University OMDS | May 2026

# Load dataset 1 - Regression

In [8]:
import pandas as pd
df = pd.read_csv('homework_1.1.csv')
df.head()

,X1,X2,X3,Y
0,-0.440646,-0.390227,0.156718,-0.877671
1,-3.810099,-1.304665,-1.105117,-10.130388
2,-1.425451,-0.340049,1.115908,0.284068
3,-1.325750,0.161906,-0.254670,-1.994344
4,3.120263,1.487343,-1.164839,2.030030


# Linear regression: predict Y from X1, X2, X3

In [9]:
from sklearn.linear_model import LinearRegression

X = df[['X1', 'X2', 'X3']]
y = df['Y']

model = LinearRegression()
model.fit(X, y)

print('Coefficients:')
print('X1:', model.coef_[0])
print('X2:', model.coef_[1])
print('X3:', model.coef_[2])

Coefficients:
X1: 1.007137655075957
X2: 1.9645685948713498
X3: 2.9754885351434215


In [10]:
for col in ['X1', 'X2', 'X3']:
    single = LinearRegression()
    single.fit(df[[col]], df['Y'])
    print(f'{col} alone: {single.coef_[0]:.4f} | with all Xs: {model.coef_[list(["X1","X2","X3"]).index(col)]:.4f}')

X1 alone: 1.8418 | with all Xs: 1.0071
X2 alone: 4.0836 | with all Xs: 1.9646
X3 alone: 3.0970 | with all Xs: 2.9755


In [11]:
import numpy as np
from sklearn.metrics import r2_score

y_pred = model.predict(X)
residuals = y - y_pred
print('t-statistics:')
for i, col in enumerate(['X1', 'X2', 'X3']):
    print(f'{col}: {model.coef_[i] / (np.std(residuals) / np.sqrt(len(df))):.4f}')

t-statistics:
X1: 63.5806
X2: 124.0232
X3: 187.8425


# Load dataset 2 - Matching

In [12]:
df2 = pd.read_csv('homework_1.2.csv')
df2.head()

,X,Y,Z
0,0,0.548814,0.548814
1,1,1.215189,0.715189
2,0,0.602763,0.602763
3,0,0.544883,0.544883
4,0,0.423655,0.423655


# NearestNeighbors best match

In [15]:
from sklearn.neighbors import NearestNeighbors

# Separate treatment and control groups
treated = df2[df2['X'] == 1]
control = df2[df2['X'] == 0]

# Fit NeareastNeighbors on control group Z values
nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[['Z']])

# Find best match for each treated unit
distances, indices = nn.kneighbors(treated[['Z']])

print('Farthest match distance:', distances.max())

Farthest match distance: 0.2102170871093757


In [16]:
# Get the matched control Y values
matched_control = control.iloc[indices.flatten()]

# Calculate the effect
effect = treated['Y'].mean() - matched_control['Y'].mean()
print('Effect:', effect)

Effect: 0.5433600652185839


In [17]:
from sklearn.neighbors import RadiusNeighborsClassifier

# Use radius_neighbors to find all matches within 0.2
rnn = NearestNeighbors(radius=0.2)
rnn.fit(control[['Z']])

distances_r, indices_r = rnn.radius_neighbors(treated[['Z']])

# Count duplicates (all but first in each group)
all_indices = []
for idx_list in indices_r:
    all_indices.extend(idx_list)

from collections import Counter
counts = Counter(all_indices)
duplicates = sum(v - 1 for v in counts.values() if v > 1)
print('Number of duplicates:', duplicates)

Number of duplicates: 685


# Radius matching within 0.2

In [18]:
# For each treated unit, get mean Y of its radius neighbors
effect_list = []
for i, idx_list in enumerate(indices_r):
    if len(idx_list) > 0:
        matched_y = control.iloc[idx_list]['Y'].mean()
        treated_y = treated.iloc[i]['Y']
        effect_list.append(treated_y - matched_y)

effect_radius = np.mean(effect_list)
print('Effect (radius matching):', effect_radius)

Effect (radius matching): 0.5688516534127853


In [19]:
# Alternative: mean of matched control Y per treated unit
control_means = []
treated_means = []

for i, idx_list in enumerate(indices_r):
    if len(idx_list) > 0:
        control_means.append(control.iloc[idx_list]['Y'].mean())
        treated_means.append(treated.iloc[i]['Y'])

effect_radius2 = np.mean(treated_means) - np.mean(control_means)
print('Effect (radius matching v2):', effect_radius2)

Effect (radius matching v2): 0.5688516534127853


In [20]:
print('Z min:', df2['Z'].min())
print('Z max:', df2['Z'].max())
print('Z range:', df2['Z'].max() - df2['Z'].min())
print('Farthest match as % of range:', 0.2102 / (df2['Z'].max() - df2['Z'].min()) * 100)

Z min: 0.004695476
Z max: 0.9883738380592262
Z range: 0.9836783620592262
Farthest match as % of range: 21.368773382385744
